# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and is accessible here: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

> **Note**: All entities such as record sets, fields, and columns are referenced by their Croissant `@id` throughout this notebook to ensure traceability and reproducibility.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print overview: metadata.name and metadata.description
md = dataset.metadata
print(md.name + ': ' + md.description)

## 2. Data Overview
List all available record sets, along with their field `@id`s, and show basic information about their schema.

Below we iterate through all available record sets, and for each we print its human-readable name, its `@id`, and its field structure.

In [ ]:
# Discover all available record sets by their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    # Fallback for datasets that define record sets differently (older or different schema)
    print("No record sets available in the dataset metadata. Please check the schema definition.")
else:
    for rs in record_sets:
        print(f"Record set: {rs.name} @id: {rs['@id']}")
        print("Fields:")
        for f in getattr(rs, 'field', []):
            print(f"  - {getattr(f, 'name', None)} (@id: {f['@id']})   dataType: {getattr(f, 'dataType', None)}")
        print('-'*36)

> **Note:** For this exploration, we will load and process the main tabular clinical record set. All field and record set references will be by `@id` as required.

## 3. Data Extraction
We will now extract the data for each record set, loading the records as pandas DataFrames.

First, list all record set `@id`s. Then we extract each as a separate DataFrame, so you can analyze or merge them further. To ensure reproducibility, use the `@id` (not just the name) to refer to each entity.

In [ ]:
# Collect all record set @id's
record_sets = dataset.metadata.recordSet
all_recordsets_ids = [rs['@id'] for rs in record_sets]
print("Available record set @ids:")
for rs in all_recordsets_ids:
    print(rs)

# Create DataFrames for each record set
dataframes = {}
for rs_id in all_recordsets_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Print columns of the first record set
main_rs_id = all_recordsets_ids[0] if all_recordsets_ids else None
if main_rs_id:
    print(f"Main record set @id: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found to load records.")

## 4. Exploratory Data Analysis (EDA)

Now, let's perform exploratory data analysis. We'll select a numeric field (by its `@id`), filter for interesting ranges or outliers, normalize it for comparability, and group by a categorical field (e.g., sex or diagnosis) to compute means/aggregate statistics.

> **Tip:** Always use precise `@id` when referring to fields for filtering/grouping.

In [ ]:
# For demonstration, let's select the main record set and pick numeric/categorical fields from its schema.
main_df = dataframes[main_rs_id].copy() if main_rs_id else None

# Inspect columns and use an example numeric field and group field by their @id
# Please substitute these with actual field @id's present in your dataset as needed.
example_numeric_field = None
example_group_field = None

# Guess numeric and group fields using metadata (if possible)
if main_rs_id and main_df is not None:
    fields = None
    for rs in dataset.metadata.recordSet:
        if rs['@id'] == main_rs_id:
            fields = getattr(rs, 'field', [])
            break
    possible_numeric = [f['@id'] for f in fields if getattr(f, 'dataType', '') in ['schema:Integer', 'schema:Float', 'schema:Number']]
    possible_group = [f['@id'] for f in fields if getattr(f, 'dataType', '') in ['schema:Text', 'schema:Boolean']]
    if possible_numeric:
        example_numeric_field = possible_numeric[0]
    if possible_group:
        example_group_field = possible_group[0]
    print(f"Using numeric field: {example_numeric_field}")
    print(f"Using group field: {example_group_field}")
else:
    print("Main DataFrame not available.")

# Filter rows by a threshold on the numeric field
if main_df is not None and example_numeric_field in main_df.columns and pd.api.types.is_numeric_dtype(main_df[example_numeric_field]):
    threshold = main_df[example_numeric_field].quantile(0.75)
    filtered_df = main_df[main_df[example_numeric_field] > threshold].copy()
    print(f"Filtered records with {example_numeric_field} > {threshold} (75th percentile):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[example_numeric_field + '_normalized'] = (
        (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
    )
    print(f"Normalized {example_numeric_field} (z-score):")
    display(filtered_df[[example_numeric_field, example_numeric_field + '_normalized']].head())

    # Grouping by group field and calculating mean of the numeric field
    if example_group_field and example_group_field in filtered_df.columns:
        grouped = (
            filtered_df.groupby(example_group_field)[example_numeric_field].mean().to_frame('mean_' + example_numeric_field)
        )
        print(f"Grouped mean of {example_numeric_field} by {example_group_field}:")
        display(grouped.head())
else:
    print(f"Numeric field '{example_numeric_field}' not available or not numeric.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and the group mean, with matplotlib/seaborn.

> Adjust field `@id`s as needed for your dataset. Re-run EDA above if you change fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and example_numeric_field and example_numeric_field in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[example_numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.ylabel("Count")
    plt.show()

    if example_group_field and example_group_field in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=main_df[example_group_field], y=main_df[example_numeric_field])
        plt.title(f"{example_numeric_field} by {example_group_field}")
        plt.xlabel(example_group_field)
        plt.ylabel(example_numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization could not be created because the required fields are missing.")

## 6. Conclusion

- We loaded the FAIR² clinical oncology dataset described in Croissant format using `mlcroissant`.
- Explored available record sets and field structure using entity `@id`s.
- Extracted a table using its record set `@id` and demonstrated numeric EDA and grouping using valid field `@id`s.
- Plotted distributions and groupings for exploratory review.
> This workflow ensures traceability, reproducibility, and compliance with FAIR data standards using the Croissant model and `mlcroissant` Python library.